# Predictive task training

In [1]:
import sys
import os

project_path = "/home/sagemaker-user/gbm_hackathon"
if project_path not in sys.path:
    sys.path.append(project_path)
    print(sys.path)

In [2]:
%load_ext autoreload
%autoreload 2
import os, sys
import pandas as pd
import numpy as np
import pickle as pkl
import torch
import seaborn as sns
import matplotlib.pyplot as plt 
import subprocess

from foundation.clinical import get_batch
from gbmhackathon.data import MosaicDataset
from gbmhackathon.s3_loader import load_s3, write_s3

In [3]:
%load_ext autoreload
%autoreload 2

from gbmhackathon.training.predictive import *
from gbmhackathon.models.mme import GBMNet
from gbmhackathon.utils.loss_functions import InfoNCELoss, RegularizedInfoNCELoss, SmoothingFunction, RankMe
from gbmhackathon.utils.module_functions import instantiate
from gbmhackathon.s3_loader import load_s3

import os
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# To investigate gradients
from torchviz import make_dot
from sklearn.ensemble import GradientBoostingClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
torch.set_num_threads(12)
torch.get_num_threads()

12

In [5]:
device = "cpu" #"cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)

In [6]:
name_emb_dict = {"hne":"embeddings_HnE_OptimusH0.pkl",
#"spatial":"2025-03-23_18-32_spatial_emb_V1.pkl",
"clinical":"2025-03-30_14-23_clinical_emb_V1.pkl",
"wes":"2025-04-05_13-40_wes_emb_V1.pkl",
"bulk":"2025-05-03_10-15_bulk_emb_V1.pkl",
"scRNA":"2025-05-04_02-35_scRNA_emb_V1.pkl"}
pkl_storage_folder = "embedding_V1"

In [7]:
dataset = PredictiveLearningDataset(name_emb_dict, pkl_storage_folder, device=device, dropout=0.0)
print(f"Dataset size: {len(dataset)}")
BATCH_SIZE = 144
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_predictive, generator=torch.Generator(device=dataset.device))


cas 2 : device reconnu : cpu 
Using device: cpu
Dataset size: 114


In [8]:
toy_batch = next(iter(dataloader))
#toy_batch

In [9]:
type(toy_batch)

tuple

In [10]:
toy_batch[2]['hne'].size()

torch.Size([114, 1536])

In [11]:
raw_emb = toy_batch[2]
inpute_size_dict = {}
for mod in raw_emb.keys():
    print(mod, raw_emb[mod].size())
    inpute_size_dict[mod] = raw_emb[mod].size(1)

hne torch.Size([114, 1536])
clinical torch.Size([114, 12])
wes torch.Size([114, 1790])
bulk torch.Size([114, 3072])
scRNA torch.Size([114, 3072])


In [12]:
raw_emb = toy_batch[2]

# 1) collecter les tenseurs et construire input_size_dict
tensors_to_concat = []
input_size_dict = {}

for mod, emb in raw_emb.items():
    if isinstance(emb, torch.Tensor):
        # on suppose que emb a la forme [batch_size, feature_size]
        input_size_dict[mod] = emb.size(1)
        tensors_to_concat.append(emb)
    else:
        # si tu veux voir ce qu'on skip :
        print(f"Skip {mod}: not a Tensor but {type(emb)}")

# 2) concaténation
# attention tous les emb doivent avoir le même batch_size (dim 0)
concat_emb = torch.cat(tensors_to_concat, dim=1)

print("Sizes by module:", input_size_dict)
print("Concatenated tensor size:", concat_emb.size())

Sizes by module: {'hne': 1536, 'clinical': 12, 'wes': 1790, 'bulk': 3072, 'scRNA': 3072}
Concatenated tensor size: torch.Size([114, 9482])


In [13]:
X=concat_emb

In [14]:
Y=toy_batch[-2]

In [15]:
y_reg=Y[:,:3]
y_cat=Y[:,3:]

In [16]:
print(y_reg.size(),y_cat.size())

torch.Size([114, 3]) torch.Size([114, 2])


In [17]:
print(X.size(),Y.size())

torch.Size([114, 9482]) torch.Size([114, 5])


In [23]:
from sklearn.multioutput import MultiOutputRegressor, MultiOutputClassifier
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error, f1_score
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier


# X : features, y_reg : array shape (n_samples,3), y_cat : array shape (n_samples,2)
loo = LeaveOneOut()
rmse_sums = np.zeros(3)
f1_sums = np.zeros(2)
n = X.shape[0]

for i, (train_idx, test_idx) in enumerate(loo.split(X), 1):
    
    
#for train_idx, test_idx in loo.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_reg_train, y_reg_test = y_reg[train_idx], y_reg[test_idx]
    y_cat_train, y_cat_test = y_cat[train_idx], y_cat[test_idx]

    # Meta-estimateurs multitâche
    #reg_mt = MultiOutputRegressor(GradientBoostingRegressor())       # régression 3 sorties :contentReference[oaicite:3]{index=3}
    #clf_mt = MultiOutputClassifier(GradientBoostingClassifier())      # classification 2 sorties :contentReference[oaicite:4]{index=4}


    reg_mt = MultiOutputRegressor(
        HistGradientBoostingRegressor(max_iter=100, early_stopping=True),
        n_jobs=-1)
    clf_mt = MultiOutputClassifier(
        HistGradientBoostingClassifier(max_iter=100, early_stopping=True),
        n_jobs=-1)

    
    # Entraînement
    reg_mt.fit(X_train, y_reg_train)
    clf_mt.fit(X_train, y_cat_train)

    # Prédiction
    y_reg_pred = reg_mt.predict(X_test)
    y_cat_pred = clf_mt.predict(X_test)

    # Accumulation des métriques
    for k in range(3):
        rmse_sums[k] += np.sqrt(mean_squared_error(y_reg_test[:, k], y_reg_pred[:, k]))
    for j in range(2):
        f1_sums[j] += f1_score(y_cat_test[:, j], y_cat_pred[:, j], average='macro')
        
    print(f"Iteration {i}/{n}", end="\r")
    
# Moyennes LOO
rmse_means = rmse_sums / n
f1_means = f1_sums / n

print("Multitâche LOO → RMSE:", rmse_means, "— F1:", f1_means)


Multitâche LOO → RMSE: [0.75757657 0.70900914 0.75248359] — F1: [0.81578947 0.85087719]


In [19]:
X.shape

torch.Size([114, 9482])

In [18]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import numpy as np
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import f1_score



In [19]:
class TabularDataset(Dataset):
    """
    Dataset PyTorch pour des données tabulaires.
    X: array-like de shape (n_samples, n_features)
    Y: array-like de shape (n_samples, 5) où les 3 premières colonnes sont continues,
       et les 2 dernières sont catégorielles (entiers).
    """
    def __init__(self, X, Y, device='cpu'):
        self.X = torch.from_numpy(np.asarray(X)).float()
        self.Y = torch.from_numpy(np.asarray(Y))
        self.device = device

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y_cont = self.Y[idx, :3].float()
        y_cat = self.Y[idx, 3:].long()
        return x.to(self.device), y_cont.to(self.device), y_cat.to(self.device)


def get_loo_loaders(X, Y, batch_size=32, device='cpu'):
    """
    Renvoie un itérateur de folds Leave-One-Out.
    Pour chaque itération, on obtient train_loader, val_loader.
    """
#    cpu_count = os.cpu_count() or 1
    dataset = TabularDataset(X, Y, device=device)
    loo = LeaveOneOut()
    for train_idx, val_idx in loo.split(X):
        train_subset = Subset(dataset, train_idx)
        val_subset = Subset(dataset, val_idx)

        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_subset, batch_size=1, shuffle=False)

        
#        train_loader = DataLoader(
#            train_subset,
#            batch_size=batch_size,
#            shuffle=True,
#            num_workers=cpu_count,  # parallélisation du chargement
#            pin_memory=(device!='cpu')  # épingle en mémoire pour GPU
#        )
#        val_loader = DataLoader(
#            val_subset,
#            batch_size=1,
#            shuffle=False,
#            num_workers=cpu_count//2,
#            pin_memory=(device!='cpu')
#        )
        yield train_loader, val_loader

In [20]:
def eval_epoch(model, loader, device):
    """
    Évaluation sans rétropropagation. Calcule RMSE et F1 comme train_epoch,
    mais passe en mode eval et n'appelle pas backward/step.
    """
    model.eval()
    se_sums = [0.0, 0.0, 0.0]
    n_samples = 0
    all_preds_a, all_labels_a = [], []
    all_preds_b, all_labels_b = [], []

    criterion_mse = nn.MSELoss(reduction='sum')
    criterion_ce = nn.CrossEntropyLoss()

    for X_batch, y_cont, y_cat in loader:
        o1, o2, o3, logits_a, logits_b = model(X_batch)

        y1, y2, y3 = y_cont[:, 0], y_cont[:, 1], y_cont[:, 2]
        y_a, y_b = y_cat[:, 0], y_cat[:, 1]

        loss1 = criterion_mse(o1.squeeze(), y1)
        loss2 = criterion_mse(o2.squeeze(), y2)
        loss3 = criterion_mse(o3.squeeze(), y3)
        loss4a = criterion_ce(logits_a, y_a)
        loss4b = criterion_ce(logits_b, y_b)

        batch_size = X_batch.size(0)
        n_samples += batch_size
        se_sums[0] += loss1.item()
        se_sums[1] += loss2.item()
        se_sums[2] += loss3.item()

        preds_a = torch.argmax(logits_a, dim=1).cpu().numpy()
        preds_b = torch.argmax(logits_b, dim=1).cpu().numpy()
        all_preds_a.extend(preds_a)
        all_preds_b.extend(preds_b)
        all_labels_a.extend(y_a.cpu().numpy())
        all_labels_b.extend(y_b.cpu().numpy())

    rmse1 = np.sqrt(se_sums[0] / n_samples)
    rmse2 = np.sqrt(se_sums[1] / n_samples)
    rmse3 = np.sqrt(se_sums[2] / n_samples)
    f1_a = f1_score(all_labels_a, all_preds_a, average='macro')
    f1_b = f1_score(all_labels_b, all_preds_b, average='macro')

    return {'rmse1': rmse1, 'rmse2': rmse2, 'rmse3': rmse3, 'f1_a': f1_a, 'f1_b': f1_b}



In [21]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    se_sums = [0.0, 0.0, 0.0]
    n_samples = 0
    all_preds_a, all_labels_a = [], []
    all_preds_b, all_labels_b = [], []

    criterion_mse = nn.MSELoss(reduction='sum')
    criterion_ce = nn.CrossEntropyLoss()

    n_batches = len(loader)
    for batch_idx, (X_batch, y_cont, y_cat) in enumerate(loader, 1):
        # Affichage de l'avancement par batch
        #print(f"  Batch {batch_idx}/{n_batches}", end="\r")

        optimizer.zero_grad()
        o1, o2, o3, logits_a, logits_b = model(X_batch)

        y1, y2, y3 = y_cont[:, 0], y_cont[:, 1], y_cont[:, 2]
        y_a, y_b = y_cat[:, 0], y_cat[:, 1]

        loss1 = criterion_mse(o1.squeeze(), y1)
        loss2 = criterion_mse(o2.squeeze(), y2)
        loss3 = criterion_mse(o3.squeeze(), y3)
        loss4a = criterion_ce(logits_a, y_a)
        loss4b = criterion_ce(logits_b, y_b)

        loss = loss1 + loss2 + loss3 + loss4a + loss4b
        loss.backward()
        optimizer.step()

        batch_size = X_batch.size(0)
        n_samples += batch_size
        se_sums[0] += loss1.item()
        se_sums[1] += loss2.item()
        se_sums[2] += loss3.item()

        preds_a = torch.argmax(logits_a, dim=1).cpu().numpy()
        preds_b = torch.argmax(logits_b, dim=1).cpu().numpy()
        all_preds_a.extend(preds_a)
        all_preds_b.extend(preds_b)
        all_labels_a.extend(y_a.cpu().numpy())
        all_labels_b.extend(y_b.cpu().numpy())

    rmse1 = np.sqrt(se_sums[0] / n_samples)
    rmse2 = np.sqrt(se_sums[1] / n_samples)
    rmse3 = np.sqrt(se_sums[2] / n_samples)
    f1_a = f1_score(all_labels_a, all_preds_a, average='macro')
    f1_b = f1_score(all_labels_b, all_preds_b, average='macro')

    return {'rmse1': rmse1, 'rmse2': rmse2, 'rmse3': rmse3, 'f1_a': f1_a, 'f1_b': f1_b}


def fit_loo(model, X, Y, optimizer, device, num_epochs=20, batch_size=32, scheduler=None):
    """
    Entraîne le modèle avec LOOCV et affiche des statistiques globales.
    Affiche uniquement deux lignes actualisables : une pour le fold courant et une pour l'epoch.
    """
    history = []
    total_folds = len(X)
    # Préparation des listes pour statistiques globales
    for fold, (train_loader, val_loader) in enumerate(
            get_loo_loaders(X, Y, batch_size=batch_size, device=device), start=1):
        # Affichage du fold courant, actualisable
        print(f"Fold {fold}/{total_folds}", end='')
        for epoch in range(1, num_epochs + 1):
            # Réécrit la ligne pour l'epoch en cours
            print(f" | Epoch {epoch}/{num_epochs}", end='', flush=True)
            train_metrics = train_epoch(model, train_loader, optimizer, device)
            val_metrics = eval_epoch(model, val_loader, device)
            if scheduler:
                scheduler.step(val_metrics['rmse1'])
        # À la fin des epochs, on passe à la ligne suivante
        print()
        history.append({'fold': fold, 'train': train_metrics, 'val': val_metrics})

    # Calcul des moyennes sur tous les folds
    all_rmse1 = [h['val']['rmse1'] for h in history]
    all_rmse2 = [h['val']['rmse2'] for h in history]
    all_rmse3 = [h['val']['rmse3'] for h in history]
    all_f1_a  = [h['val']['f1_a']   for h in history]
    all_f1_b  = [h['val']['f1_b']   for h in history]

    avg_rmse1 = np.mean(all_rmse1)
    avg_rmse2 = np.mean(all_rmse2)
    avg_rmse3 = np.mean(all_rmse3)
    avg_f1_a  = np.mean(all_f1_a)
    avg_f1_b  = np.mean(all_f1_b)

    # Affichage final des moyennes globales
    print("=== Moyennes sur tous les folds ===")
    print(f"Val RMSEs moyens : [{avg_rmse1:.4f}, {avg_rmse2:.4f}, {avg_rmse3:.4f}]")
    print(f"Val F1 moyens : a={avg_f1_a:.4f}, b={avg_f1_b:.4f}")
    return history


In [22]:
dataset = TabularDataset(X, Y, device=device)
loader = DataLoader(dataset, batch_size=16, shuffle=True)
x_batch, y_cont_batch, y_cat_batch = next(iter(loader))
print(f"Batch shapes: X={x_batch.shape}, y_cont={y_cont_batch.shape}, y_cat={y_cat_batch.shape}")

Batch shapes: X=torch.Size([16, 9482]), y_cont=torch.Size([16, 3]), y_cat=torch.Size([16, 2])


In [23]:

class SimpleMLP(nn.Module):
    """
    MLP multitâche avec architecture configurable via `layers`.
    `layers` est une liste d'entiers [in_dim, h1, h2, ..., hN].
    """
    def __init__(self, layers, n_cat_a, n_cat_b):
        super().__init__()
        modules = []
        for in_dim, out_dim in zip(layers[:-1], layers[1:]):
            modules.append(nn.Linear(in_dim, out_dim))
            modules.append(nn.ReLU())
        self.shared = nn.Sequential(*modules)
        last = layers[-1]
        self.out1 = nn.Linear(last, 1)
        self.out2 = nn.Linear(last, 1)
        self.out3 = nn.Linear(last, 1)
        self.clf_a = nn.Linear(last, n_cat_a)
        self.clf_b = nn.Linear(last, n_cat_b)

    def forward(self, x):
        h = self.shared(x)
        return (
            self.out1(h),
            self.out2(h),
            self.out3(h),
            self.clf_a(h),
            self.clf_b(h),
        )




In [ ]:
# Instanciation du modèle avec layers
layers = [X.shape[1], 1024, 512, 64]
model = SimpleMLP(layers=layers, n_cat_a=int(Y[:,3].max().item())+1, n_cat_b=int(Y[:,4].max().item())+1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Lancer une itération LOO pour vérifier
history = fit_loo(model, X, Y, optimizer, device, num_epochs=100, batch_size=16)
#print(history)


In [ ]:

#history = fit_loo(model, X, Y, optimizer, device, num_epochs=100, batch_size=16)


history = fit_loo(model, X, Y, optimizer, device, num_epochs=100, batch_size=16)
=== Moyennes sur tous les folds ===
Val RMSEs moyens : [0.0497, 0.0432, 0.0551]
Val F1 moyens : a=0.9912, b=0.9912